<a href="https://colab.research.google.com/github/tsubasa-iino/psi4book/blob/main/compchem_book_ch04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4章 分子をコンピュータで扱う方法について知ろう

### 環境構築

#### Google Colab上にPsi4をインストール

In [2]:
!pip install -q condacolab
import condacolab
import os

# バグ回避パッチ
if "LD_LIBRARY_PATH" not in os.environ:
    os.environ["LD_LIBRARY_PATH"] = ""

print("Installing CondaColab (Base)...")
condacolab.install()

Installing CondaColab (Base)...
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


ランタイム｜セッションを再起動する｜はい を実行。

In [ ]:
import condacolab
import os
import sys

condacolab.check()

# 1. 邪魔なPinningを削除
if os.path.exists("/usr/local/conda-meta/pinned"):
    !rm /usr/local/conda-meta/pinned

# 2. Python 3.12 と Psi4 をインストール（ディスク書き換え）
print("Upgrading Python to 3.12 & Installing Psi4...")
!mamba install -y -q python=3.12 psi4 -c conda-forge/label/libint_dev -c conda-forge

# 3. Pinningの復元（成功環境の再現）
os.makedirs("/usr/local/conda-meta", exist_ok=True)
with open("/usr/local/conda-meta/pinned", "w") as f:
    f.write("python 3.12.*\n")

# 4. 【最重要】カーネルの自殺（強制再起動）
# これにより、メモリ上のPython 3.11を殺し、ディスク上のPython 3.12をロードさせます
print("\n🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...")
import time
time.sleep(1)
os.kill(os.getpid(), 9)

✨🍰✨ Everything looks OK!
Upgrading Python to 3.12 & Installing Psi4...
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...


ランタイム｜セッションを再起動する｜はい を実行。

In [1]:
import sys
import os

# パスが通っていなければ通す
target_path = "/usr/local/lib/python3.12/site-packages"
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import psi4
print(f"✅ Restart Successful.")
print(f"Psi4 Version: {psi4.__version__}")
print(f"Python Version: {sys.version.split()[0]}") # ここが3.12になっているはず

# 計算テスト
psi4.set_memory('500 MB')
mol = psi4.geometry("O\nH 1 0.96\nH 1 0.96 2 104.5")
en = psi4.energy('scf/cc-pvdz')
print(f"Energy: {en:.6f}")

✅ Restart Successful.
Psi4 Version: 1.10
Python Version: 3.12.12
Energy: -76.026633


ここまででインストール確認完了。

In [10]:
import os
import datetime
import numpy as np
import pandas as pd
import psi4

print(f'current time: {datetime.datetime.now()}')
print(f'python version:\n{sys.version}')
print(f'numpy version: {np.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'psi4 version: {psi4.__version__}')

current time: 2026-01-13 05:27:16.022624
python version:
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy version: 2.0.2
pandas version: 2.2.2
psi4 version: 1.10


#### 計算資源の設定

In [11]:
# 計算資源の確認（CPU, RAM）
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
address sizes

In [12]:
!cat /proc/meminfo

MemTotal:       13286956 kB
MemFree:         7451068 kB
MemAvailable:   11964120 kB
Buffers:          212464 kB
Cached:          4250196 kB
SwapCached:            0 kB
Active:          1415788 kB
Inactive:        3873268 kB
Active(anon):       2668 kB
Inactive(anon):   826992 kB
Active(file):    1413120 kB
Inactive(file):  3046276 kB
Unevictable:           8 kB
Mlocked:               8 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:               312 kB
Writeback:             0 kB
AnonPages:        826336 kB
Mapped:           544852 kB
Shmem:              3256 kB
KReclaimable:     379480 kB
Slab:             440100 kB
SReclaimable:     379480 kB
SUnreclaim:        60620 kB
KernelStack:        5464 kB
PageTables:        14580 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6643476 kB
Committed_AS:    3101268 kB
VmallocTotal:   34359738367 kB
VmallocUsed:       11944 kB
VmallocChunk:    

In [19]:
n_cpu = os.cpu_count()

In [20]:
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)

In [21]:
# 環境に応じて計算資源を設定
psi4.set_num_threads(n_cpu)
psi4.set_memory(f'{ram * 0.9: .0f}GB')

11000000000

### Psi4のMoleculeオブジェクトについて知ろう

In [22]:
# Moleculeオブジェクトの作成
h2o = psi4.geometry('''
0 1
O       -0.1176269719      0.7387773605      0.0000000000
H        0.8523730281      0.7387773605      0.0000000000
H       -0.4409567836      1.4770262439     -0.5397651517
''')

In [23]:
type(h2o)

psi4.core.Molecule

#### オブジェクトの複製

In [24]:
h2o_2 = h2o.clone()

In [25]:
h2o == h2o_2

False

In [26]:
print(h2o.save_string_xyz())

0 1
 O    0.000000000000    0.000000000000   -0.062675830412
 H    0.792000605218    0.000000000000    0.497355455604
 H   -0.792000605218   -0.000000000000    0.497355455604



In [27]:
print(h2o_2.save_string_xyz())

0 1
 O    0.000000000000    0.000000000000   -0.062675830412
 H    0.792000605218    0.000000000000    0.497355455604
 H   -0.792000605218   -0.000000000000    0.497355455604



#### 代表的なメソッドの使い方

In [28]:
print(f'molecular charge: {h2o.molecular_charge()}')
print(f'multiplicity: {h2o.multiplicity()}')
print(f'number of atoms: {h2o.natom()}')

molecular charge: 0
multiplicity: 1
number of atoms: 3


In [29]:
bohr2ang = psi4.constants.bohr2angstroms

In [30]:
for i in range(h2o.natom()):
    x = h2o.x(i) * bohr2ang
    y = h2o.y(i) * bohr2ang
    z = h2o.z(i) * bohr2ang

    print(f'{h2o.symbol(i)}\t{x:.3f}\t{y:.3f}\t{z:.3f}')

O	0.000	0.000	-0.063
H	0.792	0.000	0.497
H	-0.792	-0.000	0.497


In [31]:
print(h2o.save_string_xyz())

0 1
 O    0.000000000000    0.000000000000   -0.062675830412
 H    0.792000605218    0.000000000000    0.497355455604
 H   -0.792000605218   -0.000000000000    0.497355455604



In [32]:
print(h2o.save_string_xyz_file())

3

 O    0.000000000000    0.000000000000   -0.062675830412
 H    0.792000605218    0.000000000000    0.497355455604
 H   -0.792000605218   -0.000000000000    0.497355455604



### py3Dmolで分子を描画しよう

In [33]:
!pip install py3Dmol
import py3Dmol

In [34]:
def show_3D(mol: psi4.core.Molecule) -> py3Dmol.view:
    """
    Psi4のMoleculeオブジェクトを描画する
    Args:
        mol: 描画対象の分子
    Return:
        py3Dmol.view: py3Dmolの描画オブジェクト

    """
    view = py3Dmol.view(width=400, height=400)
    xyz = mol.save_string_xyz_file()
    view.addModel(xyz, 'xyz')
    view.setStyle({'stick': {}})
    view.setBackgroundColor('#e1e1e1')
    view.zoomTo()

    return view.show()

In [35]:
show_3D(h2o)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.